# Tutorial 10 — Benchmarking and evaluation

[Lecture notes](../lectures/10_evaluation.md) · [Slides](../slides/10_evaluation.pptx) · [Exercise sheet](../exercises/10_evaluation.md)

**40 minutes, CPU, NumPy only.** You will score probabilities, check calibration, watch a leaderboard flatter its winner, and compute the metrics the four capstone tasks are ranked on. Every number below comes from invented or simulated data. Nothing here measures a real model.

| Minutes | Step |
|---|---|
| 0–10 | Sections 1–3: Brier score, reliability, abstention, a judge audit |
| 10–15 | Section 4: the best of 50 equal systems |
| 15–20 | Section 5: a paired test (McNemar) |
| 20–30 | Sections 6–8: micro and macro F1, greedy matching, within-painting correlation |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Accuracy and proper scoring

Ten invented binary predictions. `p` is the probability the model gives to class 1. Accuracy looks only at which side of 0.5 each `p` falls. The Brier score also charges for how far `p` is from the label.

Before running: count the errors by eye and predict the accuracy.

In [2]:
y=np.array([1,0,1,0,1,0,1,0,1,0])
p=np.array([.95,.8,.7,.1,.4,.2,.6,.55,.9,.35])
pred=(p>=.5).astype(int); conf=np.maximum(p,1-p); correct=(pred==y)
print("accuracy",correct.mean(),"binary Brier",np.mean((p-y)**2))
assert np.isclose(np.mean((np.array([.8,.7,.4,.1])-np.array([1,0,1,0]))**2),.225)

accuracy 0.7 binary Brier 0.17375000000000002


Three of the ten predictions are wrong, and the worst is the second: 0.8 on a negative example costs 0.64 of the Brier total by itself. Which single prediction would you change to cut the Brier score most without changing accuracy?

## 2. Reliability and selective prediction

Bins refer to confidence in the predicted class, `max(p, 1−p)`. Empty bins are left out of ECE and printed with count zero. The second loop answers only when confidence passes a threshold.

Before running: predict what happens to coverage and to selective accuracy as the threshold rises.

In [3]:
edges=np.linspace(0,1,6); ece=0
for j,(lo,hi) in enumerate(zip(edges[:-1],edges[1:])):
    mask=(conf>=lo)&((conf<hi) if j<4 else (conf<=hi))
    if mask.any():
        acc=correct[mask].mean(); c=conf[mask].mean()
        ece+=mask.mean()*abs(acc-c)
        print("bin",lo,hi,"n",mask.sum(),"accuracy",acc,"confidence",c)
    else: print("bin",lo,hi,"n",0)
print("ECE",ece)
for threshold in [.5,.7,.9,.99]:
    keep=conf>=threshold
    print("threshold",threshold,"coverage",keep.mean(),"selective accuracy",float(correct[keep].mean()) if keep.any() else None)

bin 0.0 0.2 n 0
bin 0.2 0.4 n 0
bin 0.4 0.6000000000000001 n 3 accuracy 0.3333333333333333 confidence 0.5833333333333334
bin 0.6000000000000001 0.8 n 2 accuracy 1.0 confidence 0.675
bin 0.8 1.0 n 5 accuracy 0.8 confidence 0.8700000000000001
ECE 0.17500000000000004
threshold 0.5 coverage 1.0 selective accuracy 0.7
threshold 0.7 coverage 0.6 selective accuracy 0.8333333333333334
threshold 0.9 coverage 0.3 selective accuracy 1.0
threshold 0.99 coverage 0.0 selective accuracy None


With ten examples each bin holds two to five items, so one flipped label moves a bin's accuracy by 20 points or more. At threshold 0.9 the model answers two questions and gets both right: perfect accuracy on 20% coverage. At 0.99 it answers nothing.

The next cell uses 5,000 simulated predictions, enough to fill ten bins. Two models share the same decisions. The honest one reports the true probability. The overconfident one reports a sharpened version of it.

In [4]:
def ece_score(conf, correct, bins=10):
    edges = np.linspace(0.5, 1, bins + 1); idx = np.minimum(((conf - 0.5) / 0.5 * bins).astype(int), bins - 1)
    return sum((idx == b).mean() * abs(correct[idx == b].mean() - conf[idx == b].mean()) for b in range(bins) if (idx == b).any())

z = rng.normal(0, 1.5, 5000)
p_true = 1 / (1 + np.exp(-z)); y_big = (rng.random(5000) < p_true).astype(int)
p_over = 1 / (1 + np.exp(-2.5 * z))                      # same sign as z, so the same decisions
for name, q in [("honest", p_true), ("overconfident", p_over)]:
    c = np.maximum(q, 1 - q); ok = ((q >= 0.5) == y_big)
    print(f"{name:14s} accuracy {ok.mean():.3f}  Brier {np.mean((q - y_big) ** 2):.3f}  ECE {ece_score(c, ok):.3f}")
conf_big = np.maximum(p_over, 1 - p_over); ok_big = ((p_over >= 0.5) == y_big)
for threshold in [0.5, 0.9, 0.99]:
    keep = conf_big >= threshold
    print(f"overconfident, threshold {threshold}: coverage {keep.mean():.2f}  selective accuracy {ok_big[keep].mean():.3f}")
assert np.isclose(((p_true >= 0.5) == y_big).mean(), ok_big.mean())

honest         accuracy 0.732  Brier 0.178  ECE 0.009
overconfident  accuracy 0.732  Brier 0.198  ECE 0.130
overconfident, threshold 0.5: coverage 1.00  selective accuracy 0.732
overconfident, threshold 0.9: coverage 0.55  selective accuracy 0.836
overconfident, threshold 0.99: coverage 0.22  selective accuracy 0.908


Both models have accuracy 0.732, because they make the same decisions. The overconfident one has ECE 0.130 against 0.009, and a worse Brier score. When it says 0.99 or more it is right 91% of the time. A user who trusts that 0.99 is misled on one answer in eleven.

## 3. Audit order-sensitive judging

A simulated judge compares systems A and B on six items, then sees the same six pairs with the display order swapped. The arrays record which system it preferred. The last lines bootstrap the accuracy from section 1 by resampling items.

Before running: count how many of the six verdicts change.

In [5]:
normal=np.array(["A","A","B","A","B","A"])
swapped=np.array(["B","A","A","B","B","A"])
print("identity preference changes after swap",np.mean(normal!=swapped))
assert np.mean(normal!=swapped)==.5
# Paired uncertainty for accuracy uses evaluation items, not probability bins.
idx=rng.integers(0,len(y),size=(1000,len(y)))
print("accuracy bootstrap interval",np.quantile(correct[idx].mean(axis=1),[.025,.975]))

identity preference changes after swap 0.5
accuracy bootstrap interval [0.4 1. ]


Half the verdicts flip when only the order changes, so on those items the judge was scoring position. The bootstrap interval on ten items is 0.4 to 1.0. Would you rank two systems on this test set?

## 4. How a leaderboard flatters its winner

Fifty systems have the same true accuracy, 0.70. Each is scored on the same 200-item test set and the best score goes on the leaderboard. Then the winner is scored once on 200 fresh items. The simulation repeats this 2,000 times.

Before running: predict the winner's average leaderboard score and its average score on fresh items.

In [6]:
n_items, n_systems, trials, true_acc = 200, 50, 2000, 0.70
board = rng.binomial(n_items, true_acc, size=(trials, n_systems)) / n_items
winner_board = board.max(1)
winner_fresh = rng.binomial(n_items, true_acc, size=trials) / n_items
print(f"winner on the leaderboard: {winner_board.mean():.3f}   same system on fresh items: {winner_fresh.mean():.3f}")
print(f"standard error of one score: {np.sqrt(true_acc * (1 - true_acc) / n_items):.3f}")
assert winner_board.mean() > 0.74 and abs(winner_fresh.mean() - true_acc) < 0.01

winner on the leaderboard: 0.771   same system on fresh items: 0.699
standard error of one score: 0.032


The winner shows 0.771 on the leaderboard and 0.699 on fresh items. No system is better than any other: the 7-point gap is the luck of the best of 50 draws, a little over two standard errors of one score. How many development runs did your team compare last week, and on how many items?

## 5. A paired test for two runs on the same items

Two runs scored on the same items differ only on the items where exactly one of them is right. Call those counts `b` (first right, second wrong) and `c` (first wrong, second right). If the runs were equally good, each such item would be a fair coin. The one-sided exact McNemar test asks how often a fair coin gives `b` or more heads in `b + c` tosses.

Task 12 uses this test as its image-use check: the first run sees the real painting, the second sees a decoy painting with the same caption.

In [7]:
from math import comb

def mcnemar_one_sided(b, c):
    """P(X >= b) for X ~ Binomial(b + c, 1/2): the chance of this many wins if the two runs were equal."""
    n = b + c
    return sum(comb(n, k) for k in range(b, n + 1)) / 2 ** n

for b, c in [(15, 5), (12, 8), (60, 40)]:
    print(f"b={b:3d} c={c:3d}  net gain {b - c:3d} items  p = {mcnemar_one_sided(b, c):.4f}")
assert np.isclose(mcnemar_one_sided(15, 5), 21700 / 2 ** 20)

b= 15 c=  5  net gain  10 items  p = 0.0207
b= 12 c=  8  net gain   4 items  p = 0.2517
b= 60 c= 40  net gain  20 items  p = 0.0284


A net gain of 10 items out of 20 disagreements passes at p = 0.021. A net gain of 4 out of 20 gives p = 0.25 and fails. The items on which both runs agree do not enter the test, so a Task 12 system that ignores the image has few disagreements and no way to pass.

## 6. Micro and macro F1 for multi-label data (Task 9)

A paragraph can carry several frames, so each frame is its own yes/no decision. The table below gives true positives, false positives and false negatives for three frames. Frame A is common and frames B and C are rare.

Micro-F1 pools the counts over frames, then computes one F1. Macro-F1 computes F1 per frame, then averages.

Before running: which of the two will be higher here?

In [8]:
counts = {"A": (40, 10, 10), "B": (2, 2, 6), "C": (1, 0, 3)}       # frame: (TP, FP, FN)
f1 = lambda tp, fp, fn: 2 * tp / (2 * tp + fp + fn)
for name, (tp, fp, fn) in counts.items():
    print(f"frame {name}: TP {tp:2d} FP {fp:2d} FN {fn:2d}  F1 {f1(tp, fp, fn):.3f}")
TP, FP, FN = np.array(list(counts.values())).sum(0)
micro = f1(TP, FP, FN); macro = np.mean([f1(*v) for v in counts.values()])
print(f"micro-F1 {micro:.3f}   macro-F1 {macro:.3f}")
assert micro > macro

frame A: TP 40 FP 10 FN 10  F1 0.800
frame B: TP  2 FP  2 FN  6  F1 0.333
frame C: TP  1 FP  0 FN  3  F1 0.400
micro-F1 0.735   macro-F1 0.511


Micro-F1 is 0.735 and macro-F1 is 0.511. Frame A supplies 40 of the 43 true positives, so micro-F1 sits close to frame A's 0.800. Macro-F1 gives the two rare frames two thirds of the weight. Task 9 ranks on micro-F1 and reports macro-F1 as secondary: a system that never predicts B or C loses little on the first and a great deal on the second.

## 7. Greedy matching, the core of BERTScore (Task 6)

BERTScore embeds every token of the candidate answer and of the reference, then matches greedily. Recall is the mean, over reference tokens, of the best cosine with any candidate token. Precision is the same with the roles swapped. F1 is their harmonic mean.

The real metric uses contextual embeddings from a pretrained model. This toy uses one fixed random vector per word. The two city names share a common component, so their cosine is about 0.7: that overlap is built in by hand, to show what it does to the score.

The question is "Mji mkuu wa Tanzania ni upi?" and the reference answer is "Dodoma".

In [9]:
d = 64
base = {w: rng.normal(size=d) for w in ["Dodoma", "Nairobi", "mji", "wa", "ndizi"]}
city = rng.normal(size=d)
for w in ["Dodoma", "Nairobi"]:
    base[w] = base[w] * 0.65 + city                       # city names share a component
emb = {w: v / np.linalg.norm(v) for w, v in base.items()}
print("cosine(Dodoma, Nairobi) =", round(float(emb["Dodoma"] @ emb["Nairobi"]), 2),
      " cosine(Dodoma, ndizi) =", round(float(emb["Dodoma"] @ emb["ndizi"]), 2))

def greedy_prf(candidate, reference):
    S = np.array([[emb[r] @ emb[c] for c in candidate.split()] for r in reference.split()])   # reference x candidate
    recall, precision = S.max(1).mean(), S.max(0).mean()
    return precision, recall, 2 * precision * recall / (precision + recall)

for cand in ["Dodoma", "mji wa Dodoma", "Nairobi", "ndizi"]:
    P, R, F = greedy_prf(cand, "Dodoma")
    print(f"{cand:15s} precision {P:.2f}  recall {R:.2f}  F1 {F:.2f}")
assert np.isclose(greedy_prf("Dodoma", "Dodoma")[2], 1)

cosine(Dodoma, Nairobi) = 0.68  cosine(Dodoma, ndizi) = 0.03
Dodoma          precision 1.00  recall 1.00  F1 1.00
mji wa Dodoma   precision 0.34  recall 1.00  F1 0.51
Nairobi         precision 0.68  recall 0.68  F1 0.68
ndizi           precision 0.03  recall 0.03  F1 0.03


The wrong city scores 0.68, and the right answer with two extra words scores 0.51. Extra tokens lower precision, because each must find a match in a one-token reference. A wrong answer of the right type keeps a high cosine. The sizes of these gaps belong to the toy. Run the organisers' scorer on the same four answers to see the real ones, and strip filler words from your system's answers before scoring.

## 8. Within-painting correlation (Task 12, subtask 3)

For each painting, the gold data give the share of positive annotators in each language. Subtask 3 ranks systems on the Pearson correlation between predicted and gold shares across languages, computed inside each painting and then averaged over paintings.

Two simulated systems. System X knows each painting's average share and guesses the language differences. System Y has the level wrong for every painting and the language differences right.

In [10]:
gold = np.array([[0.80, 0.60, 0.70, 0.90],      # rows: paintings; columns: four languages
                 [0.30, 0.50, 0.20, 0.40],
                 [0.60, 0.55, 0.75, 0.50]])
system_x = gold.mean(1, keepdims=True) + rng.normal(0, 0.05, gold.shape)
system_y = 0.5 * gold + 0.05

def within_painting_r(pred, gold):
    return np.mean([np.corrcoef(p_row, g_row)[0, 1] for p_row, g_row in zip(pred, gold)])

for name, s in [("X: right level, guessed differences", system_x), ("Y: wrong level, right differences", system_y)]:
    print(f"{name:38s} mean abs error {np.abs(s - gold).mean():.3f}   within-painting r {within_painting_r(s, gold):.3f}")
assert np.isclose(within_painting_r(system_y, gold), 1)

X: right level, guessed differences    mean abs error 0.099   within-painting r 0.206
Y: wrong level, right differences      mean abs error 0.233   within-painting r 1.000


System X is closer to the gold shares (mean absolute error 0.099 against 0.233) and scores r = 0.21. System Y scores r = 1.00. Pearson correlation ignores level and scale inside each painting, so subtask 3 rewards knowing how languages differ on the same painting. Subtask 2 uses log loss, which does charge for the level. A system that predicts one value for every language has zero variance and no defined correlation: check how the scorer treats that case.

## 9. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Micro and macro F1 from label matrices.** `Y_true` and `Y_pred` are 0/1 arrays with one row per paragraph and one column per frame. Return `(micro, macro)`. A frame with no true and no predicted positives gets F1 = 0, and so does a micro-F1 with no positives at all.

In [ ]:
# YOUR TURN
def micro_macro_f1(Y_true, Y_pred):
    """Return (micro-F1, macro-F1) for 0/1 matrices of shape (items, labels)."""
    raise NotImplementedError("your turn")

In [12]:
Yt = np.array([[1, 0, 0], [1, 1, 0], [1, 0, 0], [0, 1, 0]])
Yp = np.array([[1, 0, 0], [1, 0, 0], [0, 0, 0], [1, 1, 0]])
mi, ma = micro_macro_f1(Yt, Yp)
assert np.isclose(mi, 2 * 3 / (2 * 3 + 1 + 2))           # pooled: TP 3, FP 1, FN 2
assert np.isclose(ma, (2 / 3 + 2 / 3 + 0) / 3)           # third frame never appears: F1 = 0
assert micro_macro_f1(np.zeros((2, 2), int), np.zeros((2, 2), int)) == (0.0, 0.0)
print(f"micro {mi:.3f}  macro {ma:.3f}")
print("A passed")

micro 0.667  macro 0.444
A passed


**B. Log loss in bits (Task 12, subtask 2).** `p` is the predicted share of positive annotators for each item and `y` is the gold share. The loss for one item is $-\,[\,y\log_2 p + (1-y)\log_2(1-p)\,]$, the cost in bits of one annotator's label. Clip `p` to [0.01, 0.99] first, then return the mean over items. The clip is this course's choice: read the organisers' scorer for theirs.

In [ ]:
# YOUR TURN
def log_loss_bits(p, y):
    """Mean of -(y*log2(p) + (1-y)*log2(1-p)) with p clipped to [0.01, 0.99]."""
    raise NotImplementedError("your turn")

In [14]:
assert np.isclose(log_loss_bits(np.array([0.5, 0.5]), np.array([0.9, 0.1])), 1.0)     # 0.5 always costs 1 bit
assert np.isclose(log_loss_bits(np.array([0.7]), np.array([0.8])), 0.759, atol=1e-3)
assert np.isclose(log_loss_bits(np.array([0.0]), np.array([1.0])), -np.log2(0.01))    # clipped, so finite
print("confident and wrong:", round(float(log_loss_bits(np.array([0.99]), np.array([0.2]))), 2), "bits")
print("gold share itself:  ", round(float(log_loss_bits(np.array([0.2]), np.array([0.2]))), 2), "bits")
print("B passed")

confident and wrong: 5.32 bits
gold share itself:   0.72 bits
B passed


## Discussion

1. Can two equally accurate models differ in calibration? Point to the output above that answers this.
2. Why must an abstention threshold be chosen on development data?
3. What would reveal position bias in a model judge?
4. Your Task 6 answer scores BERTScore F1 0.85. Name one check that tells you whether the answer is right.

## From the toy to your task

Every team now writes its metric as a function it can run on development predictions: BERTScore F1 with the organisers' model and settings (Task 6), compliance and relevance (Task 7), micro and macro F1 (Task 9), macro-F1 per language, log loss in bits or within-painting correlation (Task 12). Use the organisers' scorer for any number you report. Use your own function to understand what the scorer rewards.

Submit this notebook with both tests passing and a 150–250-word note: the metric your task is ranked on, one output it would score too generously, and how you will catch that case.